In [1]:
import anndata as ad
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import os, sys
import mudata as md
md.set_options(pull_on_update=False)

# Include src directory in the path dynamically
notebook_dir = os.path.abspath("")
src_path = os.path.join(notebook_dir, "../src")
sys.path.append(src_path)

import preprocessing.preprocessing as preprocessing
from constants import ModalityType, MsiPreprocessingParams

## Constant Definitions

In [2]:
PATH = "/mnt/data/lorenzo/FOCUS/p_LipidQMap/d_liver_VIB_Ghent/"
PLOT_PATH = os.path.join(PATH, "plots", "preprocessing")
MODALITY_NAME = "MSI"
MODALITY_TYPE = ModalityType.MSI

sc.settings.figdir = PLOT_PATH

In [3]:
from sklearn.decomposition import PCA, NMF

def visualize_spatial(clusters: np.ndarray, raster_coords: np.ndarray, cluster_colors: np.ndarray, title: str):
	'''
	Visualize the raster image of one sample, coloring each raster block (data spot) according to its cluster assignment.
	
	Parameters
	----------
	clusters : np.ndarray
		Array of shape (n_spots,) containing the cluster assignment for each data spot.
	raster_coords : np.ndarray
		Array of shape (n_spots, 2, 2) containing the raster coordinates for each data spot.
	cluster_colors : np.ndarray
		Array of shape (n_clusters,) containing the hex color code for each cluster.
	title : str
		Title for the plot.
	
	'''

	# Convert the list of cluster colors into a list of RGB values
	cluster_colors_rgb = {}
	for index in range(len(cluster_colors)):
		cluster_colors_rgb[str(index)] = tuple(int(cluster_colors[index][i:i+2], 16) for i in (1, 3, 5))

	# Generate a raster image based on the raster coordinates and cluster colors
	img = np.zeros((raster_coords[:,:,1].max()+1, raster_coords[:,:,0].max()+1, 3), dtype=np.uint8)
	for index, ((x1, y1), (x2, y2)) in enumerate(raster_coords):
		img[y1:y2, x1:x2] = cluster_colors_rgb[clusters[index]]

	# Add 30 pixel padding around the image
	pad_width = 30
	img = np.pad(img, ((pad_width, pad_width), (pad_width, pad_width), (0, 0)), mode='constant', constant_values=0)

	# Display the image
	plt.figure(figsize=(15,15))
	plt.imshow(img, cmap='gray')
	plt.title(title)
	plt.show()


def visualize_spot_spatial(
        clusters: np.ndarray,
        physical_coords: np.ndarray,
        cluster_colors: np.ndarray,
		title: str,
		max_coords_value: tuple[float, float] = None
        ):
    '''
    Visualize the spatial scatter plot of sample spots, coloring each spot by cluster assignment.
    
    Parameters
    ----------
    clusters : np.ndarray
        Array of shape (n_spots,) containing cluster assignments (possibly strings).
    physical_coords : np.ndarray
        Array of shape (n_spots, 2) with (x, y) coordinates.
    cluster_colors : np.ndarray
        Array of shape (n_clusters,) of hex color strings.
    title : str
        Plot title.
    max_coords_value : tuple[float, float], optional
        If provided, sets the x and y axis limits to (0, max_value). Otherwise, axes are auto-scaled.
        If provided, the limit is increased by 5% for better visualization.
    '''
    def hex_to_rgb(hex_str):
        return tuple(int(hex_str[i:i+2], 16) / 255. for i in (1, 3, 5))

    cluster_colors_rgb = [hex_to_rgb(c) for c in cluster_colors]
    unique_clusters = sorted(np.unique(clusters))
    cluster_to_idx = {c: i for i, c in enumerate(unique_clusters)}
    cluster_indices = np.array([cluster_to_idx[c] for c in clusters])

    spot_colors = [cluster_colors_rgb[i] for i in cluster_indices]

    coords = np.asarray(physical_coords, dtype=float)
    centroid = coords.mean(axis=0)

    # Center the coordinates to plot_center
    if max_coords_value is not None:
        plot_center = np.array([max_coords_value[0] / 2., max_coords_value[1] / 2.])
    else:
        bbox_min = coords.min(axis=0)
        bbox_max = coords.max(axis=0)
        plot_center = (bbox_min + bbox_max) / 2.

    # Translate the coordinates: centroid -> plot_center
    translated = coords - centroid + plot_center

    plt.figure(figsize=(12, 12))
    plt.scatter(translated[:, 0], translated[:, 1], c=spot_colors, s=2)
    plt.title(title)
    plt.xlabel("Physical X")
    plt.ylabel("Physical Y")
    plt.gca().set_aspect('equal', adjustable='box')

    # Limits: if max_coords_value is provided, center on plot_center with a small margin
    if max_coords_value is not None:
        half = np.array([max_coords_value[0] / 2., max_coords_value[1] / 2.]) * 1.05
        plt.xlim(plot_center[0] - half[0], plot_center[0] + half[0])
        plt.ylim(plot_center[1] - half[1], plot_center[1] + half[1])
    else:
        pad = (translated.max(axis=0) - translated.min(axis=0)) * 0.05
        plt.xlim(translated[:, 0].min() - pad[0], translated[:, 0].max() + pad[0])
        plt.ylim(translated[:, 1].min() - pad[1], translated[:, 1].max() + pad[1])

    # Flip y-axis for correct orientation
    plt.gca().invert_yaxis()

    plt.show()
    plt.savefig(os.path.join(PLOT_PATH, title.replace(" ", "_") + ".png"), dpi=300)
    plt.close()

## MSI Dataset preparation and processing

In [4]:
preprocessing_settings = {
    MsiPreprocessingParams.FORCE_RECOMPUTING: False,
    MsiPreprocessingParams.FREQUENCY_THRESHOLD: 0.1,
    MsiPreprocessingParams.INTENSITY_NORMALIZATION: preprocessing.MsiIntensityNormalization.TIC,
    MsiPreprocessingParams.LIPID_ANNOTATION_DB: os.path.join(PATH, "resources", "MSI_database_POS_NEG_combined.json"),
    MsiPreprocessingParams.MASS_TOLERANCE: 10,
    MsiPreprocessingParams.RECALIBRATION_REFERENCE: None,
    MsiPreprocessingParams.MIN_INTENSITY_THRESHOLD: 1e4,
    MsiPreprocessingParams.DETECT_BACKGROUND: False
}

In [ ]:
processed_msi = preprocessing.preprocess_modality(
    path=PATH,
    modality_type=MODALITY_TYPE,
    modality_name=MODALITY_NAME,
    preprocessing_settings=preprocessing_settings,
)

All samples have already been processed and merged dataset exists. Using cached results.


## Dataset Visualization

In [25]:
# Load the merged dataset
#msi_dataset = ad.read_h5ad(processed_msi['merged'], backed='r')
msi_dataset = ad.read_h5ad("/mnt/data/lorenzo/FOCUS/p_LipidQMap/d_liver_VIB_Ghent/merged/preprocessing/MSI_merged_processed.h5ad", backed='r')

In [43]:
# Add Treatment Labels
for sample_id in msi_dataset.obs['sample_id'].unique():
    if "cda" in sample_id.lower():
        msi_dataset.obs.loc[msi_dataset.obs['sample_id'] == sample_id, 'treatment'] = 'CDA4 - Recovery'
    elif "sd" in sample_id.lower():
        msi_dataset.obs.loc[msi_dataset.obs['sample_id'] == sample_id, 'treatment'] = 'SD - Control'
    elif "pla" in sample_id.lower():
        msi_dataset.obs.loc[msi_dataset.obs['sample_id'] == sample_id, 'treatment'] = 'PLA - Active Disease'

In [6]:
print(f"MSI dataset has {msi_dataset.n_obs} spots and {msi_dataset.n_vars} m/z features.")

print(f"Dataset contains {msi_dataset.obs['sample_id'].nunique()} samples.")

ion_mode_count = msi_dataset.var['mz_mode'].value_counts()
print(f"The features are: {ion_mode_count.to_dict()}")

lipid_annotation_counts = msi_dataset.var['lipid_annotation'].value_counts()
annotated_lipids = lipid_annotation_counts.drop('Unannotated', errors='ignore').sum()
print(f"Number of unique annotated lipids: {annotated_lipids}/{msi_dataset.n_vars}")

foreground_spots = msi_dataset.obs['foreground'].value_counts()
print(f"Number of foreground spots: {foreground_spots.get(True, 0)}/{msi_dataset.n_obs}")

MSI dataset has 606997 spots and 2073 m/z features.
Dataset contains 13 samples.
The features are: {'neg': 2073}
Number of unique annotated lipids: 2073/2073
Number of foreground spots: 606997/606997


In [ ]:
# For each sample, compute leiden clustering on the foreground spots and visualize the results
for sample_id in msi_dataset.obs['sample_id'].unique():
    sample_data = msi_dataset[msi_dataset.obs['sample_id'] == sample_id].copy()

    # Keep only foreground spots
    sample_data = sample_data[sample_data.obs['foreground'] == True].copy()

    # Create color map based on leiden clusters
    colors = sc.pl.palettes.vega_20 
    n_clusters = sample_data.obs['leiden'].nunique()
    cluster_colors = colors * (n_clusters // len(colors) + 1)
    sample_data.uns['leiden_colors'] = cluster_colors[:n_clusters]

    visualize_spot_spatial(
        clusters=sample_data.obs['leiden'].to_numpy(),
        physical_coords=sample_data.obsm['spatial'],
        cluster_colors=sample_data.uns['leiden_colors'],
        title=f"Leiden Clustering (Resolution=0.5) for Sample {sample_id} - Spot View",
    )

In [7]:
# Compute PCA and UMAP visualizations
sc.pp.pca(msi_dataset, n_comps=50, layer='X_tic')
sc.pp.neighbors(msi_dataset)
sc.tl.umap(msi_dataset)

In [ ]:
# Visualize the dataset
sc.pl.umap(msi_dataset, color=['sample_id'], title="MSI Dataset - Sample ID", save="_msi_dataset_sample_id.png")
sc.pl.umap(msi_dataset, color=['treatment'], title="MSI Dataset - Condition", save="_msi_dataset_condition.png")